<a href="https://colab.research.google.com/github/engmodu/AIFEL_quest_eng/blob/main/LLM_Application/LLM01/Day1_RAG_Code_QA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

last modified date : 2026.03.15  
제작 : 박광석 (모두의연구소)

# 랭체인으로 RAG 시작하기

해당 노트는 Langchain으로 RAG를 구현하기 위해 필요한
각 컴포넌트인 Document Loaders, Text splitters, Text embeddings, Vectorstores, Retriever를 다룹니다  




### Step 0 : 설치와 준비  
Langchain 설치 및 Gemini API 키를 등록하도록 합니다.  

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -U -q langchain langchain-openai
!pip install -U -q langchain-community langchain-core
!pip install -U langchain-text-splitters


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.2/121.2 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.6/99.6 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 236.4/236.4 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 82.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 64.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 6.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [3]:
import os


In [9]:
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_KEY')

In [10]:
! curl ipinfo.io

{
  "ip": "35.186.157.41",
  "hostname": "41.157.186.35.bc.googleusercontent.com",
  "city": "Singapore",
  "region": "Singapore",
  "country": "SG",
  "loc": "1.2897,103.8501",
  "org": "AS396982 Google LLC",
  "postal": "018989",
  "timezone": "Asia/Singapore",
  "readme": "https://ipinfo.io/missingauth"
}

In [12]:
from langchain_openai import ChatOpenAI

# OpenAI API를 사용하는 설정으로 변경
# 모델명은 필요에 따라 "gpt-4o", "gpt-4-turbo", "gpt-3.5-turbo" 등으로 바꿀 수 있습니다.
llm = ChatOpenAI(
    model="gpt-5",
    temperature=0.0,
)

In [13]:
!pip install -q pypdf pdf2image docx2txt pdfminer unstructured #의존성 모듈을 설치합니다

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 100.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 62.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 109.4/109.4 kB 8.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.9/343.9 kB 29.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 70.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.7/107.7 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 35.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 453.8/453.8 kB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.8/167.8 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

### Step 1 : Document Loaders 사용해보기  

Document Loader는 다양한 형태의 원본 데이터를  
LLM이 이해할 수 있는 Document 객체(text + metadata) 로 변환하는 역할을 합니다.

PDF, 웹페이지, CSV와 같이 형식이 서로 다른 문서들을 일관된 구조로 파싱하여, 이후 Chunking·Embedding·검색(Retrieval) 단계에서
바로 사용할 수 있도록 만들어줍니다.

즉, Document Loader는
**RAG 파이프라인의 가장 첫 단계에서 “데이터를 읽을 수 있는 형태로 정리하는 역할을 담당**합니다.

공식 문서에서는 지원되는 다양한 Loader 목록을 확인할 수 있습니다.
https://python.langchain.com/docs/modules/data_connection/document_loaders/

#### PDFLoader 사용  
이번 실습에서는 가장 많이 사용되는 문서 형식인 PDF 파일을 대상으로
PyPDFLoader를 사용해 문서를 불러옵니다.

실습을 위해, 질의응답에 활용하고 싶은 PDF 파일을 먼저 Colab 환경(또는 Drive)에 업로드해주세요.

PDFLoader는 각 페이지를 하나의 Document 단위로 변환하며,
이 단계에서 생성된 문서들은 이후 Text Splitter를 통해 의미 단위로 다시 분할됩니다.

In [15]:
from langchain_community.document_loaders import PyPDFLoader
file_path = "/content/drive/MyDrive/Colab Notebooks/data/Demian.pdf"
home_path = "/content/drive/MyDrive/Colab Notebooks"
loader = PyPDFLoader(home_path+"/data/Demian.pdf")
pages = loader.load_and_split()

print("현재 작업 경로 :", os.getcwd())
print("파일 존재 여부 :", os.path.exists(file_path))
print("파일 여부 :", os.path.isfile(file_path))

현재 작업 경로 : /content
파일 존재 여부 : True
파일 여부 : True
폴더 여부 : False


In [20]:
pages[0]

Document(metadata={'producer': 'Adobe Acrobat Standard DC 19 Paper Capture Plug-in', 'creator': 'ScanFix(TM) Enhanced', 'creationdate': '2015-09-10T01:40:29+00:00', 'moddate': '2019-01-30T17:47:47+01:00', 'source': '/content/drive/MyDrive/Colab Notebooks/data/Demian.pdf', 'total_pages': 182, 'page': 0, 'page_label': '1'}, page_content='DEMIAN \n• \nDownloaded from https://www.holybooks.com')

In [21]:
print(pages[10])

page_content='TWO WOR.LDS 
Finally, out of sheer nervousness, I began to talk. I 
invented a long story of robbery, in which I featured as 
the hero. One night in the comer by the mill a friend 
and I ha.d stolen a whole sackful of apples, not just 
ordinary apples but pippins, golden pippins of the best 
kind at that. I was taking refuge in my story from the 
dangers of the moment and found no difficulty in invent­
ing and relating it. In order not to dry up too soon and 
perhaps become involved in something worse, I gave full 
rein to my narrative powers. One of us, I reported, had 
always stood guard while the other sat in the tree and 
chucked the apples down, and the sack had got so heavy 
that in the end we had to open it and leave half behind, 
but we came back half an hour later and fetched them 
too. 
I hoped for some applause at the end of my story; I 
had warmed up to the narrative aJ: last, carried away by 
my own eloquence. The two smaller boys were silent, 
waiting, Lut F

출력 결과를 보기 쉽게 확인하기 위해,
Document 객체 전체가 아닌 실제 텍스트 본문이 담긴 page_content만 선택하여 확인해보겠습니다.

In [22]:
print(pages[10].page_content)

TWO WOR.LDS 
Finally, out of sheer nervousness, I began to talk. I 
invented a long story of robbery, in which I featured as 
the hero. One night in the comer by the mill a friend 
and I ha.d stolen a whole sackful of apples, not just 
ordinary apples but pippins, golden pippins of the best 
kind at that. I was taking refuge in my story from the 
dangers of the moment and found no difficulty in invent­
ing and relating it. In order not to dry up too soon and 
perhaps become involved in something worse, I gave full 
rein to my narrative powers. One of us, I reported, had 
always stood guard while the other sat in the tree and 
chucked the apples down, and the sack had got so heavy 
that in the end we had to open it and leave half behind, 
but we came back half an hour later and fetched them 
too. 
I hoped for some applause at the end of my story; I 
had warmed up to the narrative aJ: last, carried away by 
my own eloquence. The two smaller boys were silent, 
waiting, Lut Franz Kromer ga

#### CSVLoader

SV 파일은 행(row) 단위로 구조화된 데이터를 담고 있는 형식으로,
LangChain의 CSVLoader를 사용하면 각 행을 하나의 Document 객체로 변환할 수 있습니다.

이렇게 변환된 문서들은 이후 PDF나 웹 문서와 동일하게
Embedding, VectorStore, Retrieval 단계에서 함께 활용할 수 있습니다.

실습을 위해, CSV 파일을 먼저 Colab 환경(또는 Drive)에 업로드해주세요.

In [23]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader(home_path+"/data/titanic.csv")

data = loader.load()

print("파일 존재 여부 :", os.path.exists(file_path))
print("파일 여부 :", os.path.isfile(file_path))

파일 존재 여부 : True
파일 여부 : True


In [24]:
data[:3]

[Document(metadata={'source': '/content/drive/MyDrive/Colab Notebooks/data/titanic.csv', 'row': 0}, page_content='PassengerId: 1\nSurvived: 0\nPclass: 3\nName: Braund, Mr. Owen Harris\nSex: male\nAge: 22\nSibSp: 1\nParch: 0\nTicket: A/5 21171\nFare: 7.25\nCabin: \nEmbarked: S'),
 Document(metadata={'source': '/content/drive/MyDrive/Colab Notebooks/data/titanic.csv', 'row': 1}, page_content='PassengerId: 2\nSurvived: 1\nPclass: 1\nName: Cumings, Mrs. John Bradley (Florence Briggs Thayer)\nSex: female\nAge: 38\nSibSp: 1\nParch: 0\nTicket: PC 17599\nFare: 71.2833\nCabin: C85\nEmbarked: C'),
 Document(metadata={'source': '/content/drive/MyDrive/Colab Notebooks/data/titanic.csv', 'row': 2}, page_content='PassengerId: 3\nSurvived: 1\nPclass: 3\nName: Heikkinen, Miss. Laina\nSex: female\nAge: 26\nSibSp: 0\nParch: 0\nTicket: STON/O2. 3101282\nFare: 7.925\nCabin: \nEmbarked: S')]

#### 웹베이스로더  
웹베이스 로더는 웹페이지에 포함된 텍스트 콘텐츠를 직접 파싱하여 Document 객체로 변환하는 역할을 합니다.  
이를 통해 뉴스 기사, 블로그 글, 공지사항과 같은 실시간으로 업데이트되는 웹 문서를 RAG 시스템의 지식 소스로 활용할 수 있습니다.  
이번 실습에서는 실제 뉴스 기사를 예제로 사용하여,
웹페이지의 내용을 불러오고 텍스트 형태로 변환하는 과정을 살펴봅니다.  

실습에 사용할 웹페이지는 다음과 같습니다.  
https://it.chosun.com/news/articleView.html?idxno=2023092111831

In [25]:
from langchain_community.document_loaders import WebBaseLoader

In [27]:
loader = WebBaseLoader("https://it.chosun.com/news/articleView.html?idxno=2023092111831")
documents = loader.load()

print(documents[0].page_content)





































모두의연구소 ‘AI학교 아이펠’, ICLR 2024 워크숍 혁신 기술 논문 채택 < 일반 < 기업 < 기사본문 - IT조선











 






















































주요서비스 바로가기
본문 바로가기
매체정보 바로가기
로그인 바로가기
기사검색 바로가기
전체서비스 바로가기




























 






전체메뉴



기사검색








기사검색


검색

닫기














기사검색









기사검색


검색

닫기











로그인




facebook

post
youtube



UPDATED. 2026-05-27 12:00 (수) 




기사검색









기업

일반
모바일·가전
방송·통신
반도체·디스플레이
SW·보안
중공업·에너지
소부장·스타트업
유통·쇼핑
프롭테크·부동산



모빌리티

자동차·모빌리티
로봇·드론·항공
방산



게임·콘텐츠

게임·인터넷
메타버스·VR
키덜트
미디어·엔터



과학·헬스

과학·우주
의학·정책
제약·바이오



파이낸스

금융
증권
핀테크·블록체인



칼럼·인터뷰

칼럼
기고
인터뷰



알림

알립니다
인사
부음
보도자료



컴퓨팅·AI

일반
코딩
에듀테크
테크리포트



GLOBAL


GLOBAL
기획·연재
속보
백과사전











속보




기업


일반


모바일·가전


방송·통신


반도체·디스플레이


SW·보안


중공업·에너지


소부장·스타트업


유통·쇼핑


프롭테크·부동산




모빌리티


자동차·모빌리티


로봇·드론·항공


방산




게임·콘텐츠


게임·인터넷


메타버스·VR


키덜트


미디어·엔터




과학·헬스


과학·우주


의학·정책


제약·바이오




파이낸스


금융


증권


핀테크·블록체인




칼럼·인터뷰


칼럼


기고

주석을 해제하고 코드를 실행하면,
해당 웹페이지에 포함된 본문 텍스트 전체를 불러와 확인할 수 있습니다.  

웹페이지, PDF, CSV 등 서로 다른 형식의 문서들이
모두 텍스트 형태로 정상적으로 파싱된 것을 확인할 수 있습니다.  

이제 이 텍스트를 **전처리(불필요한 요소 제거, 정제)** 한 뒤,
Chunking과 Embedding 단계에 활용할 수 있습니다.  

### Step2 : TextSplitters 사용해보기  
Text Splitter는 긴 텍스트 문서를 **의미를 유지한 작은 단위(Chunk)** 로 분할하는 역할을 합니다.  
LLM은 한 번에 처리할 수 있는 토큰 수에 제한이 있기 때문에, 문서를 그대로 입력하는 대신 Splitter를 통해 분할된 여러 Chunk를 입력받아 처리하게 됩니다.  

이 과정을 통해 긴 문서에서도 토큰 길이 제약을 극복하고, 필요한 부분만 효율적으로 검색할 수 있습니다.  

분할된 각 Chunk는 이후 단계에서 1:1로 Embedding되어 VectorStore에 저장되며,
이 Chunk 단위가 RAG 시스템에서 검색과 응답의 기본 단위가 됩니다.  

In [28]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

CharacterTextSplitter는
하나의 고정된 구분자(separator)를 기준으로 텍스트를 분할하는 방식입니다.
구현이 단순하고 직관적이지만,
문서 구조에 따라 분할된 Chunk가 토큰 제한을 초과하는 경우가 발생할 수 있습니다.

반면, RecursiveCharacterTextSplitter는
줄바꿈, 문장 구분자, 구두점 등 여러 구분자를 순차적으로 적용하며
텍스트를 재귀적으로 분할합니다.

이 방식은 토큰 제한을 안정적으로 만족시키는 데 유리하지만,
분할 과정에서 의미적으로 완전하지 않은 문장 단위로 잘릴 수 있다는 단점이 있습니다.  

단순한 구조의 문서나,
문단 구성이 명확한 텍스트의 경우에는 CharacterTextSplitter로도 충분합니다.

하지만 실제 서비스 환경에서는
문서 길이와 구조가 제각각인 경우가 많기 때문에,
대부분의 RAG 시스템에서는 RecursiveCharacterTextSplitter를 기본 선택지로 사용합니다.

이는 Chunk 크기를 안정적으로 제어하면서도
검색 실패를 줄이는 데 유리하기 때문입니다.

In [30]:
with open(home_path+"/data/state_of_the_union.txt") as f:
    text = f.read()

In [31]:
#len은 어떤 기준으로 chunk size를 잴 것인가?의 기준이 되는 함수입니다.
#chunk_overlap은 chunk의 앞뒤로 다른 chunk와 설정한 size까지 겹칠 수 있도록 설정하는 것입니다.
text_splitter = CharacterTextSplitter(separator="\n\n", chunk_size=1000, chunk_overlap=100, length_function = len,)
chunks = text_splitter.split_text(text)

Chunk의 내용을 확인해보겠습니다

In [32]:
print(chunks[0])

Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans.  

Last year COVID-19 kept us apart. This year we are finally together again. 

Tonight, we meet as Democrats Republicans and Independents. But most importantly as Americans. 

With a duty to one another to the American people to the Constitution. 

And with an unwavering resolve that freedom will always triumph over tyranny. 

Six days ago, Russia’s Vladimir Putin sought to shake the foundations of the free world thinking he could make it bend to his menacing ways. But he badly miscalculated. 

He thought he could roll into Ukraine and the world would roll over. Instead he met a wall of strength he never imagined. 

He met the Ukrainian people. 

From President Zelenskyy to every Ukrainian, their fearlessness, their courage, their determination, inspires the world.


각 chunk의 길이를 확인해보겠습니다,

In [33]:
length = []
for chunk in chunks:
    length.append(len(chunk))

print(length)

[939, 995, 810, 962, 994, 883, 957, 952, 928, 915, 993, 702, 900, 950, 957, 958, 967, 996, 796, 866, 888, 966, 964, 977, 998, 948, 925, 924, 989, 965, 938, 936, 981, 965, 771, 982, 972, 977, 984, 999, 968, 801]


### 토큰 단위로 텍스트 분할해보기  
  
LLM은 문장을 단어가 아닌 토큰(token) 단위로 처리합니다.
따라서 사람이 인식하는 단어 길이나 문자 수는
실제 모델이 처리하는 입력 길이와 정확히 일치하지 않을 수 있습니다.

이로 인해 문자 수나 단어 수를 기준으로 텍스트를 분할할 경우,
모델의 입력 토큰 제한을 초과하거나
예상보다 훨씬 짧은 문맥만 전달되는 문제가 발생할 수 있습니다.

실제 서비스 환경에서는 이러한 문제를 방지하기 위해,
토큰 단위를 기준으로 텍스트를 분할하는 방식을 사용합니다.
이제 토큰 기준으로 텍스트를 분할해보겠습니다.

In [34]:
!pip install tiktoken

In [35]:
import tiktoken
tokenizer = tiktoken.get_encoding("cl100k_base")

def tiktoken_len(text):
    tokens = tokenizer.encode(
        text
    )
    return len(tokens)

In [36]:
tiktoken_length = []
for chunk in chunks:
    tiktoken_length.append(tiktoken_len(chunk))

print(length)
print(tiktoken_length)

[939, 995, 810, 962, 994, 883, 957, 952, 928, 915, 993, 702, 900, 950, 957, 958, 967, 996, 796, 866, 888, 966, 964, 977, 998, 948, 925, 924, 989, 965, 938, 936, 981, 965, 771, 982, 972, 977, 984, 999, 968, 801]
[197, 198, 163, 190, 203, 182, 195, 197, 206, 205, 218, 148, 188, 205, 216, 215, 209, 224, 176, 187, 201, 197, 201, 215, 222, 202, 203, 204, 229, 206, 184, 204, 197, 194, 156, 200, 194, 221, 203, 225, 209, 187]


글자 수와 토큰 수의 차이를 확인할 수 있습니다 !

### Step3 : TextEmbedding 사용해보기  
Embedding은 텍스트를 컴퓨터가 계산할 수 있는 수치 벡터(vector) 형태로 변환하는 과정입니다.
이 벡터는 문장의 표면적인 형태가 아니라, 의미적 유사성을 반영하도록 설계되어 있습니다.

변환된 벡터는
VectorStore에 저장되거나,
새로운 질의(Query) 벡터와의 유사도 계산을 통해
의미적으로 가까운 문서를 검색하는 데 사용됩니다.

이러한 변환은 대규모 말뭉치로 사전 학습된
Embedding 전용 모델을 통해 이루어지며,
RAG 시스템에서 Retrieval 성능을 결정하는 핵심 요소입니다.

이번 실습에서는
OpenAI 임베딩 모델을 사용해
텍스트를 벡터로 변환해보겠습니다.

In [37]:
import openai

genai 라이브러리의 list_models 함수를 사용하여 사용 가능한 모델들의 목록을 가져옵니다.

In [38]:
client = openai.OpenAI()

In [39]:
for model in client.models.list():
    if "embedding" in model.id:
        print(model.id)

text-embedding-ada-002
text-embedding-3-small
text-embedding-3-large


text-embedding-3-small은 가성비가 좋고, text-embedding-3-large는 성능이 더 강력합니다.

In [40]:
from langchain_openai import OpenAIEmbeddings


embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

만약 여러분들이 Gemini를 사용하여 구축중이시라면, embedding은 지역에 따라 사용이 제한됩니다.  
주로 유럽권에서 제한되기 때문에, 다음 에러를 확인하신다면 Colab 파일의 서버 저장 위치를 확인 후, 다른 임베딩 모델로 변경해야합니다.  

Error embedding content: 400 User location is not supported for the API use.


In [41]:
!curl ipinfo.io

{
  "ip": "35.186.157.41",
  "hostname": "41.157.186.35.bc.googleusercontent.com",
  "city": "Singapore",
  "region": "Singapore",
  "country": "SG",
  "loc": "1.2897,103.8501",
  "org": "AS396982 Google LLC",
  "postal": "018989",
  "timezone": "Asia/Singapore",
  "readme": "https://ipinfo.io/missingauth"
}

In [ ]:
# 400 User location is not supported for the API use 오류가 발생한다면, 이 블록을 대신 실행해주세요

# ! pip install -q sentence_transformers

#from langchain.embeddings import HuggingFaceEmbeddings
#embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

embedding model 변수에 OpenAI 임베딩모델 혹은 huggingface의 임베딩모델이 할당되었을 것입니다.  
embed_documents 멤버 함수를 사용하여 새 문장을 변환해보겠습니다  

In [42]:
embeddings = embedding_model.embed_documents(
    [
        "This is red apple.",
        "This is yellow banana.",
        "This is green lime.",
    ]
)

임베딩으로 잘 변환되었는지 확인해보겠습니다  

In [43]:
print(embeddings[1])

[0.006427764892578125, -0.020843505859375, -0.0216217041015625, 0.0235443115234375, -0.0428466796875, -0.004589080810546875, 0.039703369140625, 0.05267333984375, -0.0018796920776367188, -0.0237579345703125, 0.0138092041015625, 0.0159912109375, -0.044464111328125, -0.00011223554611206055, -0.007648468017578125, 0.02557373046875, 0.025421142578125, 0.03546142578125, -0.051788330078125, 0.01232147216796875, 0.02484130859375, -0.00047087669372558594, 0.0193939208984375, 0.07525634765625, -0.002044677734375, -0.009185791015625, 0.0168304443359375, -0.007129669189453125, 0.02655029296875, -0.04852294921875, 0.043182373046875, -0.042572021484375, 0.01163482666015625, -0.03289794921875, -0.0333251953125, -0.04620361328125, 0.0019054412841796875, 0.0225677490234375, -0.018096923828125, 0.03253173828125, 0.0293426513671875, 0.011749267578125, -0.01446533203125, 0.003063201904296875, 0.024322509765625, 0.04815673828125, -0.01354217529296875, 0.04937744140625, -0.0406494140625, 0.04400634765625, 0

In [44]:
len(embeddings[1])

1536

새로운 쿼리를 넣어, 임베딩끼리 유사도를 계산해보겠습니다

In [45]:
import numpy as np
from numpy import dot
from numpy.linalg import norm
def cos_sim(A, B):
  return dot(A, B)/(norm(A)*norm(B))

In [46]:
query = ["this is red fruit"]

In [47]:
e_query = embedding_model.embed_documents(query)
print(cos_sim(embeddings[0], e_query[0]))
print(cos_sim(embeddings[1], e_query[0]))
print(cos_sim(embeddings[2], e_query[0]))

0.7478929665954975
0.4898791411166917
0.4083791543048118


빨간 사과와 빨간 과일의 유사도가 많이 높게 나왔습니다!  
  
임베딩 모델은 사용 언어나 필요에 따라 다양하게 교체하여 사용할 수 있습니다.  
해당 링크에서 여러 목록을 확인하실 수 있습니다.  
https://python.langchain.com/docs/integrations/text_embedding/

### Step4 : VectorStore 사용해보기
VectorStore는 텍스트를 Embedding 모델을 통해 벡터(vector)로 변환한 뒤, 이를 저장하고 관리하는 저장소입니다.
이 저장소는 단순한 데이터 보관 공간이 아니라,
벡터 간의 유사도를 빠르게 계산하고 탐색하기 위한 인덱싱 구조를 함께 포함하고 있습니다.

문서나 쿼리가 Embedding된 이후에는,
VectorStore를 통해 의미적으로 유사한 벡터를 효율적으로 검색할 수 있으며,
이 과정이 RAG 시스템의 Retrieval 단계를 담당하게 됩니다.

대표적인 VectorStore로는
Chroma, FAISS 등이 있으며,
각각 로컬 환경과 대규모 서비스 환경에서 널리 사용됩니다.

이번 실습에서는
구성이 단순하고 로컬 환경에서 바로 사용할 수 있는
ChromaDB를 사용해 VectorStore를 구성해보겠습니다.

In [56]:
#!pip install chromadb
!pip uninstall -y opentelemetry-api opentelemetry-sdk opentelemetry-semantic-conventions
!pip install -U opentelemetry-api opentelemetry-sdk opentelemetry-semantic-conventions
!pip install -U langchain-chroma chromadb

Found existing installation: opentelemetry-api 1.42.1
Uninstalling opentelemetry-api-1.42.1:
  Successfully uninstalled opentelemetry-api-1.42.1
Found existing installation: opentelemetry-sdk 1.42.1
Uninstalling opentelemetry-sdk-1.42.1:
  Successfully uninstalled opentelemetry-sdk-1.42.1
Found existing installation: opentelemetry-semantic-conventions 0.63b1
Uninstalling opentelemetry-semantic-conventions-0.63b1:
  Successfully uninstalled opentelemetry-semantic-conventions-0.63b1
  Using cached opentelemetry_api-1.42.1-py3-none-any.whl.metadata (1.4 kB)
  Using cached opentelemetry_sdk-1.42.1-py3-none-any.whl.metadata (1.7 kB)
  Using cached opentelemetry_semantic_conventions-0.63b1-py3-none-any.whl.metadata (2.4 kB)
Using cached opentelemetry_api-1.42.1-py3-none-any.whl (61 kB)
Using cached opentelemetry_sdk-1.42.1-py3-none-any.whl (170 kB)
Using cached opentelemetry_semantic_conventions-0.63b1-py3-none-any.whl (203 kB)
ERROR: pip's dependency resolver does not currently take into ac

In [2]:
!pip install langchain-chroma

In [1]:
from langchain_chroma import Chroma

In [3]:
!pip install --upgrade opentelemetry-api
!pip install --upgrade opentelemetry-sdk

제일 처음에 사용했던, PDF를 다시 사용하도록 합니다!  

In [9]:
# 위에서 사용했던 코드입니다
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import tiktoken

def tiktoken_len(text):
    tokenizer = tiktoken.get_encoding("cl100k_base")
    tokens = tokenizer.encode(text)
    return len(tokens)

#file_path = "/content/drive/MyDrive/Colab Notebooks/data/Demian.pdf"
home_path = "/content/drive/MyDrive/Colab Notebooks"
loader = PyPDFLoader(home_path+"/data/Demian.pdf")
#loader = PyPDFLoader("/content/Demian.pdf")
pages = loader.load_and_split()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0, length_function = tiktoken_len)
docs = text_splitter.split_documents(pages)

In [10]:
!pip show chromadb

Name: chromadb
Version: 1.5.9
Summary: Chroma.
Home-page: https://github.com/chroma-core/chroma
Author: 
Author-email: Jeff Huber <jeff@trychroma.com>, Anton Troynikov <anton@trychroma.com>
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: bcrypt, build, grpcio, httpx, importlib-resources, jsonschema, kubernetes, mmh3, numpy, onnxruntime, opentelemetry-api, opentelemetry-exporter-otlp-proto-grpc, opentelemetry-sdk, orjson, overrides, pybase64, pydantic, pydantic-settings, pypika, pyyaml, rich, tenacity, tokenizers, tqdm, typer, typing-extensions, uvicorn
Required-by: langchain-chroma


In [12]:
!pip install -U langchain-openai openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 48.2 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 2.37.0
    Uninstalling openai-2.37.0:
      Successfully uninstalled openai-2.37.0


In [18]:
!pip install -U sentence-transformers

Chroma에 임베딩 시킵니다  

In [20]:
from google.colab import userdata
from langchain_openai import ChatOpenAI
import os

from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_KEY')
# OpenAI API를 사용하는 설정으로 변경
# 모델명은 필요에 따라 "gpt-4o", "gpt-4-turbo", "gpt-3.5-turbo" 등으로 바꿀 수 있습니다.
llm = ChatOpenAI(
    model="gpt-5",
    temperature=0.0,
)


from langchain_chroma import Chroma

db = Chroma.from_documents(
    docs,
    embedding_model
)

#db = Chroma.from_documents(docs, embedding_model)

print("벡터DB 생성 완료")



/tmp/ipykernel_9338/3529975205.py:7: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

벡터DB 생성 완료


이제 쿼리를 날려보겠습니다

In [21]:
query = "how Demian look like?"
docs = db.similarity_search(query)

In [22]:
print(docs[0].page_content)

DEMIAN 
Why had it only just dawned on me I It wu Demian'• 
face. 
Later I often comP9Xed the face on the paper with 
Demian's features as l remembered them. They were 
certainly, though similar, not the same. But beyond all 
doubt, it was Demian. 
Once one evening in early summer the sun was slant­
ing red through my window that faced westward. Inside 
the room it was dusk It occurred to me to attach the 
picture of Beatrice (or Demian) to the window bar and 
watch the effect as the sun shone through. The outlines 
of the face were blurred but the eyes, edged with pink., 
the brightness of the forehead and the energetic red 
mouth glowed excitingly from the surface. For a long 
time I sat opposite it even after the picture had faded 
out. And gradually a feeling came over me that it was 
neither Beatrice nor Demian but myself. Not that the 
picture was like me-I did not feel it should be-but 
the face somehow expressed my life, it was my inner self, 
my fate or my daimon. That was how

Face, features, looks like 등 데미안의 생김새를 담고 있는 페이지가 출력되었습니다  
굉장히 빠른 속도로 검색했습니다!  

### Step5 : Retriever 사용해보기  

Retriever는 사용자의 질문을 Embedding 모델을 통해 벡터로 변환한 뒤,
VectorStore에 저장된 문서 벡터들과 비교하여
의미적으로 가장 유사한 문서(Chunk)를 찾아 반환하는 역할을 합니다.

즉, Retriever는
RAG 시스템에서 “어떤 정보를 LLM에게 참고 자료로 줄 것인가”를 결정하는 핵심 컴포넌트이며,
검색 결과의 품질이 곧 최종 답변의 품질로 이어집니다.
  

In [23]:
!pip install -U langchain langchain-classic

In [24]:
from langchain_classic.chains.retrieval_qa.base import RetrievalQA


긴 문서 전체를 한 번에 LLM에 전달하는 대신,
Retriever와 LLM을 결합한 RetrievalQA 체인을 사용하여
문서에서 질문과 관련된 부분만 검색하고,
그 결과를 바탕으로 답변을 생성합니다.

이를 통해 길이가 긴 문서에서도
토큰 제한을 넘지 않으면서, 근거 기반의 질의응답을 수행할 수 있습니다.

In [27]:
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

# OpenAI 모델로 변경
# streaming=True와 callbacks 설정을 통해 실시간 출력을 활성화합니다.
llm = ChatOpenAI(
    model="gpt-4o",              # 또는 "gpt-4o-mini"
    temperature=0.0,
    streaming=True,              # 실시간 출력을 켭니다
    callbacks=[StreamingStdOutCallbackHandler()], # 출력을 콘솔에 바로 뿌려줍니다
)

체인의 종류와 검색(Retrieval) 방식,
그리고 그에 따른 주요 파라미터를 설정합니다.

이 단계에서는
Retriever가 어떤 전략으로 문서를 검색할지,
그리고 몇 개의 문서를 LLM에게 전달할지를 결정하게 됩니다.
이 선택은 최종 답변의 품질과 직접적으로 연결됩니다.

예를 들어,
MMR(Maximal Marginal Relevance) 방식은
쿼리와의 유사도뿐만 아니라 문서 간의 중복을 줄이고 다양성을 확보하는 재정렬(Re-ranking) 전략입니다.

실무 환경에서는 단일 문서에 정보가 몰리는 것을 방지하고,
LLM이 보다 풍부한 문맥을 참고하도록 하기 위해
MMR 방식이 자주 사용됩니다.

In [28]:
qa = RetrievalQA.from_chain_type(llm, chain_type="stuff",
                                 retriever=db.as_retriever(
                                     search_type="mmr",
                                     search_kwargs={"k": 3, "fetch_k" : 10}),
                                 return_source_documents=True)

위 코드에서 짚고 넘어갈 파라미터는 다음과 같습니다  
🔹 chain_type="stuff"

검색된 문서(Chunk)를 그대로 하나의 Prompt에 모두 삽입하는 방식입니다.
구조가 단순하고 이해하기 쉬워,
RAG 구조를 처음 학습하거나 프로토타입을 만들 때 적합합니다.
단점으로는 문서 수가 많아질 경우
토큰 사용량이 빠르게 증가할 수 있습니다.
실무에서는 초기 검증 단계에서는 stuff를,
문서 수가 많아지면 map_reduce나 refine 방식으로 확장합니다.  

🔹 retriever

VectorStore에서 어떤 문서를 검색할지 결정하는 검색 모듈입니다.
검색 전략과 파라미터 설정에 따라 LLM이 참고하는 정보의 범위와 품질이 달라집니다.  

🔹 search_type="mmr"

MMR(Maximal Marginal Relevance) 검색 방식을 사용합니다. 쿼리와의 유사도뿐만 아니라, 문서 간 중복을 줄여 다양한 문맥을 확보하는 Re-ranking 전략입니다. 실무 환경에서 단일 문서 편향을 줄이기 위해 자주 사용됩니다

🔹 search_kwargs={"k": 3, "fetch_k": 10}  
- fetch_k  
VectorStore에서 우선적으로 가져올 후보 문서 개수입니다. Re-ranking 이전 단계에서 사용됩니다.
- k  
최종적으로 LLM에게 전달할 문서(Chunk)의 개수입니다.

일반적으로 fetch_k > k 로 설정하여 후보 풀을 넉넉히 확보한 뒤, 품질 좋은 문서만 선별하는 방식을 사용합니다.  

🔹 return_source_documents=True

답변 생성에 사용된 원문 문서(Chunk)를 함께 반환합니다. 이를 통해 답변의 출처를 사용자에게 표시하거나 검색 품질을 디버깅하고 RAG 성능을 평가할 수 있습니다. 실무 서비스에서는 거의 필수적으로 사용하는 옵션입니다.

In [29]:
query = "how demian looks like"
result = qa(query)

/tmp/ipykernel_9338/3336337621.py:2: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  result = qa(query)


Demian is described as having a face that is somewhat wild-looking, with an interrogative and erratic expression. His features are wayward and obstinate, yet his mouth has a soft and childlike quality. The masculinity and strength are concentrated in his eyes and brow, while the lower half of his face is tender, immature, and somewhat feminine. His chin is described as irresolute and boylike, which contradicts the strength of his forehead and expression. His dark brown eyes are full of pride and humility.

마크다운 형식으로 출력해봅니다

In [41]:
from IPython.display import Markdown, display
display(Markdown(result["result"]))

Demian is described as having a face that is somewhat wild-looking, with an interrogative and erratic expression. His features are wayward and obstinate, yet his mouth has a soft and childlike quality. The masculinity and strength are concentrated in his eyes and brow, while the lower half of his face is tender, immature, and somewhat feminine. His chin is described as irresolute and boylike, which contradicts the strength of his forehead and expression. His dark brown eyes are full of pride and humility.

RAG를 사용하지 않은 llm 호출도 시도해보세요!

In [32]:
llm2 = ChatOpenAI(
    model="gpt-4o")
request = llm2.invoke("how demian looks like")
display(Markdown(request.content))


If you're referring to "Demian," the novel by Hermann Hesse, the character of Demian is not described in extraordinarily detailed physical terms. The focus is more on his influence and the philosophical ideas he represents. Demian is depicted as an enigmatic and charismatic young man with penetrating eyes that suggest a deep understanding of the world. He carries an aura of confidence and wisdom beyond his years, and his presence exerts a significant impact on the protagonist, Emil Sinclair. The emphasis in the novel is more on Demian’s role in Sinclair’s life and the ideas he embodies rather than explicit physical attributes.

### Quiz
결과의 어떤 부분을 관찰하였을 때, RAG 시스템의 결과를 신뢰할 수 있겠다 생각하셨나요?  

### Answer  
원문에서 답변의 출처를 확인할 수 있었습니다.

## 6. 완성 예제  
앞에서 진행한 내용으로, Demian을 다시 한번 읽어봅시다!  
완성하여 제출해주세요~


필요한 라이브러리를 모두 다운받습니다  

In [42]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -U -q langchain langchain-openai
!pip install -U -q langchain-community langchain-core
!pip install -U langchain-text-splitters
import os
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_KEY')

from langchain_openai import ChatOpenAI

# OpenAI API를 사용하는 설정으로 변경
# 모델명은 필요에 따라 "gpt-4o", "gpt-4-turbo", "gpt-3.5-turbo" 등으로 바꿀 수 있습니다.
llm = ChatOpenAI(
    model="gpt-5",
    temperature=0.0,
)
!pip install -q pypdf pdf2image docx2txt pdfminer unstructured #의존성 모듈을 설치합니다

from langchain_community.document_loaders import PyPDFLoader
file_path = "/content/drive/MyDrive/Colab Notebooks/data/bible_pdf/1-20잠언.pdf"
home_path = "/content/drive/MyDrive/Colab Notebooks"
loader = PyPDFLoader(home_path+"/data/bible_pdf/1-20잠언.pdf")
pages = loader.load_and_split()

print("현재 작업 경로 :", os.getcwd())
print("파일 존재 여부 :", os.path.exists(file_path))
print("파일 여부 :", os.path.isfile(file_path))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
현재 작업 경로 : /content
파일 존재 여부 : True
파일 여부 : True


In [44]:
print(pages[10].page_content)

잠14:26 여호와를 경외하는 자에게는 견고한 의뢰가 있나니 그 자녀들에게 피난처가 있으리라잠14:27 여호와를 경외하는 것은 생명의 샘이니 사망의 그물에서 벗어나게 하느니라잠14:28 백성이 많은 것은 왕의 영광이요 백성이 적은 것은 주권자의 패망이니라잠14:29 노하기를 더디 하는 자는 크게 명철하여도 마음이 조급한 자는 어리석음을 나타내느니라잠14:30 평온한 마음은 육신의 생명이나 시기는 뼈를 썩게 하느니라잠14:31 가난한 사람을 학대하는 자는 그를 지으신 이를 멸시하는 자요 궁핍한 사람을 불쌍히 여기는 자는 주를 공경하는 자니라잠14:32 악인은 그의 환난에 엎드러져도 의인은 그의 죽음에도 소망이 있느니라잠14:33 지혜는 명철한 자의 마음에 머물거니와 미련한 자의 속에 있는 것은 나타나느니라잠14:34 공의는 나라를 영화롭게 하고 죄는 백성을 욕되게 하느니라잠14:35 슬기롭게 행하는 신하는 왕에게 은총을 입고 욕을 끼치는 신하는 그의 진노를 당하느니라잠15:1 유순한 대답은 분노를 쉬게 하여도 과격한 말은 노를 격동하느니라잠15:2 지혜 있는 자의 혀는 지식을 선히 베풀고 미련한 자의 입은 미련한 것을 쏟느니라잠15:3 여호와의 눈은 어디서든지 악인과 선인을 감찰하시느니라잠15:4 온순한 혀는 곧 생명 나무이지만 패역한 혀는 마음을 상하게 하느니라잠15:5 아비의 훈계를 업신여기는 자는 미련한 자요 경계를 받는 자는 슬기를 얻을 자니라잠15:6 의인의 집에는 많은 보물이 있어도 악인의 소득은 고통이 되느니라잠15:7 지혜로운 자의 입술은 지식을 전파하여도 미련한 자의 마음은 정함이 없느니라잠15:8 악인의 제사는 여호와께서 미워하셔도 정직한 자의 기도는 그가 기뻐하시느니라잠15:9 악인의 길은 여호와께서 미워하셔도 공의를 따라가는 자는 그가 사랑하시느니라잠15:10 도를 배반하는 자는 엄한 징계를 받을 것이요 견책을 싫어하는 자는 죽을 것이니라잠15:11 스올과 아바돈도 여호와의 앞에 드러나거든 하물며 사람의 마음이리요잠15:12 거만한 자는 견

Text splitter 사용을 위한 준비입니다

### Step 1 Document loader

In [46]:
file_path = home_path + "/data/bible_txt/2-03누가복음.txt"

encodings = ["utf-8", "cp949", "euc-kr"]

for enc in encodings:
    try:
        with open(file_path, encoding=enc) as f:
            text = f.read()

        print(f"성공 인코딩 : {enc}")
        print(text[:300])
        break

    except Exception as e:
        print(f"{enc} 실패 :", e)

utf-8 실패 : 'utf-8' codec can't decode byte 0xb4 in position 0: invalid start byte
성공 인코딩 : cp949
눅1:1 <데오빌로 각하에게> 우리 중에 이루어진 사실에 대하여
눅1:2 처음부터 목격자와 말씀의 일꾼 된 자들이 전하여 준 그대로 내력을 저술하려고 붓을 든 사람이 많은지라
눅1:3 그 모든 일을 근원부터 자세히 미루어 살핀 나도 데오빌로 각하에게 차례대로 써 보내는 것이 좋은 줄 알았노니
눅1:4 이는 각하가 알고 있는 바를 더 확실하게 하려 함이로라
눅1:5 <세례 요한의 출생을 예고하다> 유대 왕 헤롯 때에 아비야 반열에 제사장 한 사람이 있었으니 이름은 사가랴요 그의 아내는 아론의 자손이니 이름은 엘리사벳이라
눅1:6 이 


In [59]:
from langchain_text_splitters import CharacterTextSplitter
#len은 어떤 기준으로 chunk size를 잴 것인가?의 기준이 되는 함수입니다.
#chunk_overlap은 chunk의 앞뒤로 다른 chunk와 설정한 size까지 겹칠 수 있도록 설정하는 것입니다.
text_splitter = CharacterTextSplitter(separator="\n", chunk_size=1000, chunk_overlap=1000, length_function = len,)
chunks = text_splitter.split_text(text)

In [60]:
print(chunks[0])

눅1:1 <데오빌로 각하에게> 우리 중에 이루어진 사실에 대하여
눅1:2 처음부터 목격자와 말씀의 일꾼 된 자들이 전하여 준 그대로 내력을 저술하려고 붓을 든 사람이 많은지라
눅1:3 그 모든 일을 근원부터 자세히 미루어 살핀 나도 데오빌로 각하에게 차례대로 써 보내는 것이 좋은 줄 알았노니
눅1:4 이는 각하가 알고 있는 바를 더 확실하게 하려 함이로라
눅1:5 <세례 요한의 출생을 예고하다> 유대 왕 헤롯 때에 아비야 반열에 제사장 한 사람이 있었으니 이름은 사가랴요 그의 아내는 아론의 자손이니 이름은 엘리사벳이라
눅1:6 이 두 사람이 하나님 앞에 의인이니 주의 모든 계명과 규례대로 흠이 없이 행하더라
눅1:7 엘리사벳이 잉태를 못하므로 그들에게 자식이 없고 두 사람의 나이가 많더라
눅1:8 마침 사가랴가 그 반열의 차례대로 하나님 앞에서 제사장의 직무를 행할새
눅1:9 제사장의 전례를 따라 제비를 뽑아 주의 성전에 들어가 분향하고
눅1:10 모든 백성은 그 분향하는 시간에 밖에서 기도하더니
눅1:11 주의 사자가 그에게 나타나 향단 우편에 선지라
눅1:12 사가랴가 보고 놀라며 무서워하니
눅1:13 천사가 그에게 이르되 사가랴여 무서워하지 말라 너의 간구함이 들린지라 네 아내 엘리사벳이 네게 아들을 낳아 주리니 그 이름을 요한이라 하라
눅1:14 너도 기뻐하고 즐거워할 것이요 많은 사람도 그의 태어남을 기뻐하리니
눅1:15 이는 그가 주 앞에 큰 자가 되며 포도주나 독한 술을 마시지 아니하며 모태로부터 성령의 충만함을 받아
눅1:16 이스라엘 자손을 주 곧 그들의 하나님께로 많이 돌아오게 하겠음이라
눅1:17 그가 또 엘리야의 심령과 능력으로 주 앞에 먼저 와서 아버지의 마음을 자식에게, 거스르는 자를 의인의 슬기에 돌아오게 하고 주를 위하여 세운 백성을 준비하리라
눅1:18 사가랴가 천사에게 이르되 내가 이것을 어떻게 알리요 내가 늙고 아내도 나이가 많으니이다


In [61]:
length = []
for chunk in chunks:
    length.append(len(chunk))

print(length)

[948, 984, 993, 979, 974, 987, 969, 988, 985, 986, 981, 990, 989, 942, 976, 972, 967, 983, 987, 998, 992, 971, 986, 988, 988, 974, 975, 986, 979, 983, 970, 948, 966, 996, 975, 989, 973, 971, 1000, 997, 997, 956, 962, 997, 983, 989, 971, 958, 975, 986, 990, 999, 999, 987, 963, 990, 988, 974, 974, 993, 1000, 969, 972, 977, 985, 985, 974, 953, 972, 963, 982, 957, 971, 973, 967, 986, 994, 996, 979, 969, 989, 990, 982, 976, 973, 962, 993, 993, 996, 969, 980, 998, 989, 965, 988, 985, 971, 995, 994, 957, 974, 973, 974, 964, 978, 970, 975, 940, 989, 970, 989, 967, 958, 956, 978, 964, 973, 997, 967, 974, 963, 951, 967, 951, 972, 999, 982, 979, 969, 952, 993, 996, 999, 995, 985, 963, 993, 982, 969, 968, 993, 967, 969, 973, 936, 948, 940, 973, 986, 988, 996, 982, 988, 981, 946, 995, 988, 989, 995, 945, 958, 998, 963, 997, 984, 952, 985, 986, 950, 981, 961, 992, 960, 986, 969, 969, 967, 966, 979, 984, 970, 973, 985, 940, 999, 957, 964, 992, 994, 988, 970, 979, 931, 1000, 999, 976, 980, 950, 946, 9

### Step 2 Text splitters

In [62]:
!pip install tiktoken

In [63]:
import tiktoken
tokenizer = tiktoken.get_encoding("cl100k_base")

def tiktoken_len(text):
    tokens = tokenizer.encode(
        text
    )
    return len(tokens)
tiktoken_length = []
for chunk in chunks:
    tiktoken_length.append(tiktoken_len(chunk))

print(length)
print(tiktoken_length)

[948, 984, 993, 979, 974, 987, 969, 988, 985, 986, 981, 990, 989, 942, 976, 972, 967, 983, 987, 998, 992, 971, 986, 988, 988, 974, 975, 986, 979, 983, 970, 948, 966, 996, 975, 989, 973, 971, 1000, 997, 997, 956, 962, 997, 983, 989, 971, 958, 975, 986, 990, 999, 999, 987, 963, 990, 988, 974, 974, 993, 1000, 969, 972, 977, 985, 985, 974, 953, 972, 963, 982, 957, 971, 973, 967, 986, 994, 996, 979, 969, 989, 990, 982, 976, 973, 962, 993, 993, 996, 969, 980, 998, 989, 965, 988, 985, 971, 995, 994, 957, 974, 973, 974, 964, 978, 970, 975, 940, 989, 970, 989, 967, 958, 956, 978, 964, 973, 997, 967, 974, 963, 951, 967, 951, 972, 999, 982, 979, 969, 952, 993, 996, 999, 995, 985, 963, 993, 982, 969, 968, 993, 967, 969, 973, 936, 948, 940, 973, 986, 988, 996, 982, 988, 981, 946, 995, 988, 989, 995, 945, 958, 998, 963, 997, 984, 952, 985, 986, 950, 981, 961, 992, 960, 986, 969, 969, 967, 966, 979, 984, 970, 973, 985, 940, 999, 957, 964, 992, 994, 988, 970, 979, 931, 1000, 999, 976, 980, 950, 946, 9

### Step 3 Vector Empeddings

In [64]:
import openai
client = openai.OpenAI()
for model in client.models.list():
    if "embedding" in model.id:
        print(model.id)

from langchain_openai import OpenAIEmbeddings
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")
!curl ipinfo.io

text-embedding-ada-002
text-embedding-3-small
text-embedding-3-large
{
  "ip": "35.186.157.41",
  "hostname": "41.157.186.35.bc.googleusercontent.com",
  "city": "Singapore",
  "region": "Singapore",
  "country": "SG",
  "loc": "1.2897,103.8501",
  "org": "AS396982 Google LLC",
  "postal": "018989",
  "timezone": "Asia/Singapore",
  "readme": "https://ipinfo.io/missingauth"
}

In [73]:
embeddings = embedding_model.embed_documents(
    [
        "예수님",
        "마리아",
        "성령",
    ]
)
print(embeddings[1])
len(embeddings[1])

[-0.0180206298828125, 0.004985809326171875, -0.01181793212890625, -0.03887939453125, -0.06890869140625, 0.0341796875, 0.01873779296875, 0.042449951171875, -0.031646728515625, 0.00222015380859375, -0.030059814453125, 0.0294189453125, 0.005558013916015625, -0.0082855224609375, 0.004486083984375, 0.01374053955078125, -0.006565093994140625, -0.042388916015625, 0.05230712890625, -0.0149688720703125, 0.0290985107421875, 0.04473876953125, -0.0096282958984375, 0.004634857177734375, 0.00699615478515625, 0.067138671875, -0.0231781005859375, -0.0116424560546875, 0.036865234375, 0.04022216796875, 0.035064697265625, -0.03375244140625, 0.049346923828125, 0.0095977783203125, -0.00041031837463378906, 0.0017404556274414062, 0.016876220703125, -0.031280517578125, 0.01042938232421875, 0.0246429443359375, -0.04150390625, -0.005725860595703125, 0.011444091796875, 0.0027980804443359375, 0.019012451171875, 0.03570556640625, -0.031585693359375, 0.005931854248046875, 0.0231475830078125, 0.006542205810546875, -

1536

In [75]:
import numpy as np
from numpy import dot
from numpy.linalg import norm
def cos_sim(A, B):
  return dot(A, B)/(norm(A)*norm(B))

In [83]:
query = ["백성"]
e_query = embedding_model.embed_documents(query)
print(cos_sim(embeddings[0], e_query[0]))
print(cos_sim(embeddings[1], e_query[0]))
print(cos_sim(embeddings[2], e_query[0]))

0.2347460615161856
0.1543533999124861
0.35105541123620204


In [1]:
#!pip install chromadb
!pip uninstall -y opentelemetry-api opentelemetry-sdk opentelemetry-semantic-conventions
!pip install -U opentelemetry-api opentelemetry-sdk opentelemetry-semantic-conventions
!pip install -U langchain-chroma chromadb
!pip install langchain-chroma
from langchain_chroma import Chroma
!pip install --upgrade opentelemetry-api
!pip install --upgrade opentelemetry-sdk
!pip show chromadb
# 위에서 사용했던 코드입니다
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import tiktoken
from google.colab import userdata
from langchain_openai import ChatOpenAI
import os

!pip install -U langchain-openai openai
!pip install -U sentence-transformers

def tiktoken_len(text):
    tokenizer = tiktoken.get_encoding("cl100k_base")
    tokens = tokenizer.encode(text)
    return len(tokens)

from langchain_community.document_loaders import PyPDFLoader
file_path = "/content/drive/MyDrive/Colab Notebooks/data/bible_pdf/1-20잠언.pdf"
home_path = "/content/drive/MyDrive/Colab Notebooks"
loader = PyPDFLoader(home_path+"/data/bible_pdf/1-20잠언.pdf")
pages = loader.load_and_split()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0, length_function = tiktoken_len)
docs = text_splitter.split_documents(pages)

from langchain_community.embeddings import HuggingFaceEmbeddings

Found existing installation: opentelemetry-api 1.42.1
Uninstalling opentelemetry-api-1.42.1:
  Successfully uninstalled opentelemetry-api-1.42.1
Found existing installation: opentelemetry-sdk 1.42.1
Uninstalling opentelemetry-sdk-1.42.1:
  Successfully uninstalled opentelemetry-sdk-1.42.1
Found existing installation: opentelemetry-semantic-conventions 0.63b1
Uninstalling opentelemetry-semantic-conventions-0.63b1:
  Successfully uninstalled opentelemetry-semantic-conventions-0.63b1
  Using cached opentelemetry_api-1.42.1-py3-none-any.whl.metadata (1.4 kB)
  Using cached opentelemetry_sdk-1.42.1-py3-none-any.whl.metadata (1.7 kB)
  Using cached opentelemetry_semantic_conventions-0.63b1-py3-none-any.whl.metadata (2.4 kB)
Using cached opentelemetry_api-1.42.1-py3-none-any.whl (61 kB)
Using cached opentelemetry_sdk-1.42.1-py3-none-any.whl (170 kB)
Using cached opentelemetry_semantic_conventions-0.63b1-py3-none-any.whl (203 kB)
ERROR: pip's dependency resolver does not currently take into ac

/tmp/ipykernel_33038/3186609261.py:11: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [6]:
# import
import os

from google.colab import userdata

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI

# OpenAI Key
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_KEY')

# PDF 로드
from langchain_community.document_loaders import PyPDFLoader
file_path = "/content/drive/MyDrive/Colab Notebooks/data/bible_pdf/1-20잠언.pdf"
home_path = "/content/drive/MyDrive/Colab Notebooks"
loader = PyPDFLoader(home_path+"/data/bible_pdf/1-20잠언.pdf")
pages = loader.load_and_split()

print("페이지 수 :", len(pages))

# Chunk 분할
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

docs = text_splitter.split_documents(pages)

print("Chunk 수 :", len(docs))
print(docs[0].page_content[:300])

# Embedding 모델
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Chroma Vector DB 생성
db = Chroma.from_documents(
    documents=docs,
    embedding=embedding_model
)

print("Chroma DB 생성 완료")

페이지 수 : 24
Chunk 수 : 51
잠1:1 <솔로몬의 잠언> 다윗의 아들 이스라엘 왕 솔로몬의 잠언이라잠1:2 이는 지혜와 훈계를 알게 하며 명철의 말씀을 깨닫게 하며잠1:3 지혜롭게, 공의롭게, 정의롭게, 정직하게 행할 일에 대하여 훈계를 받게 하며잠1:4 어리석은 자를 슬기롭게 하며 젊은 자에게 지식과 근신함을 주기 위한 것이니잠1:5 지혜 있는 자는 듣고 학식이 더할 것이요 명철한 자는 지략을 얻을 것이라잠1:6 잠언과 비유와 지혜 있는 자의 말과 그 오묘한 말을 깨달으리라잠1:7 <젊은이에게 주는 교훈> 여호와를 경외하는 것이 지식의 근본이거늘 미련한 자는 


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Chroma DB 생성 완료


In [21]:
query = "밥먹자"
docs = db.similarity_search(query)

In [22]:
print(docs[0].page_content)

악인을 멀리 하시고 의인의 기도를 들으시느니라잠15:30 눈이 밝은 것은 마음을 기쁘게 하고 좋은 기별은 뼈를 윤택하게 하느니라잠15:31 생명의 경계를 듣는 귀는 지혜로운 자 가운데에 있느니라


### Step 4 Retrievers

In [23]:
!pip install -U langchain langchain-classic
from langchain_classic.chains.retrieval_qa.base import RetrievalQA
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
llm = ChatOpenAI(
    model="gpt-5",              # 또는 "gpt-5"
    temperature=0.0,
    streaming=True,              # 실시간 출력을 켭니다
    callbacks=[StreamingStdOutCallbackHandler()], # 출력을 콘솔에 바로 뿌려줍니다
)
qa = RetrievalQA.from_chain_type(llm, chain_type="stuff",
                                 retriever=db.as_retriever(
                                     search_type="mmr",
                                     search_kwargs={"k": 3, "fetch_k" : 10}),
                                 return_source_documents=True)

In [24]:
query = "하나님은 누구신가"
result = qa(query)
from IPython.display import Markdown, display
display(Markdown(result["result"]))

/tmp/ipykernel_33038/122034310.py:2: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  result = qa(query)


질문에 주어진 잠언 말씀들이 보여 주는 하나님은 이런 분입니다.
- 악인을 멀리하시고 의인의 기도를 들으시는 분입니다 (잠언 15:29).
- 사람의 마음을 시험하고 정련하시는 분입니다 (잠언 17:3).
- 가난한 자의 창조주이시며, 약자를 조롱하는 것을 악으로 보시고 결코 무죄하지 않게 하시는 분입니다 (잠언 17:5).
- 반역과 악을 용납하지 않으시고 마땅한 심판이 따르게 하시는 분입니다 (잠언 17:11).
- 재물을 의지하는 길은 패망으로, 의로움의 길은 생명과 번성으로 이끄시는 질서를 세우신 분입니다 (잠언 11:28).

요약하면, 잠언이 증언하는 하나님은 의와 공의를 사랑하시며, 사람의 마음 깊은 곳까지 아시는 창조주이십니다.

질문에 주어진 잠언 말씀들이 보여 주는 하나님은 이런 분입니다.
- 악인을 멀리하시고 의인의 기도를 들으시는 분입니다 (잠언 15:29).
- 사람의 마음을 시험하고 정련하시는 분입니다 (잠언 17:3).
- 가난한 자의 창조주이시며, 약자를 조롱하는 것을 악으로 보시고 결코 무죄하지 않게 하시는 분입니다 (잠언 17:5).
- 반역과 악을 용납하지 않으시고 마땅한 심판이 따르게 하시는 분입니다 (잠언 17:11).
- 재물을 의지하는 길은 패망으로, 의로움의 길은 생명과 번성으로 이끄시는 질서를 세우신 분입니다 (잠언 11:28).

요약하면, 잠언이 증언하는 하나님은 의와 공의를 사랑하시며, 사람의 마음 깊은 곳까지 아시는 창조주이십니다.

### Step 5 Question Answering

In [27]:
query = "잘사는 사람이란?"
result = qa(query)


잠언에 따르면 “잘사는 사람”은 이런 사람입니다.
- 악을 멀리하고 의로워 하나님께서 기도를 들으시는 사람(잠15:29)
- 밝은 눈과 기쁜 마음으로, 좋은 소식으로 주변을 살리는 사람(잠15:30)
- 생명을 주는 책망을 기꺼이 듣고 지혜로운 자들과 함께하는 사람(잠15:31)
- 재물을 의지하지 않고 하나님을 의지하여 푸른 잎사귀처럼 번성하는 사람(잠11:28)
- 자기 집을 해치지 않고 지혜로 세우는 사람(잠11:29)
- 자신의 한계를 인정하고 거룩하신 이를 경외하는 사람(잠30:3-4)
- 하나님의 말씀을 순전히 믿고 더하지 않는 사람(잠30:5-6)
- 거짓과 헛됨을 멀리하고, 가난도 부요도 지나치지 않게 필요한 양식으로 만족을 구하는 사람(잠30:7-9)
- 배부름으로 교만하거나 궁핍으로 범죄하지 않기를 구해 하나님의 이름을 존중하는 사람(잠30:9)
- 아랫사람을 함부로 비방하지 않는 공의로운 사람(잠30:10)

요약하면, 잘사는 사람은 의와 지혜를 좇아 정직·절제·만족을 배우며, 하나님을 의지하고 이웃을 살리는 사람입니다.

잠언에 따르면 “잘사는 사람”은 이런 사람입니다.
- 악을 멀리하고 의로워 하나님께서 기도를 들으시는 사람(잠15:29)
- 밝은 눈과 기쁜 마음으로, 좋은 소식으로 주변을 살리는 사람(잠15:30)
- 생명을 주는 책망을 기꺼이 듣고 지혜로운 자들과 함께하는 사람(잠15:31)
- 재물을 의지하지 않고 하나님을 의지하여 푸른 잎사귀처럼 번성하는 사람(잠11:28)
- 자기 집을 해치지 않고 지혜로 세우는 사람(잠11:29)
- 자신의 한계를 인정하고 거룩하신 이를 경외하는 사람(잠30:3-4)
- 하나님의 말씀을 순전히 믿고 더하지 않는 사람(잠30:5-6)
- 거짓과 헛됨을 멀리하고, 가난도 부요도 지나치지 않게 필요한 양식으로 만족을 구하는 사람(잠30:7-9)
- 배부름으로 교만하거나 궁핍으로 범죄하지 않기를 구해 하나님의 이름을 존중하는 사람(잠30:9)
- 아랫사람을 함부로 비방하지 않는 공의로운 사람(잠30:10)

요약하면, 잘사는 사람은 의와 지혜를 좇아 정직·절제·만족을 배우며, 하나님을 의지하고 이웃을 살리는 사람입니다.

In [31]:
query = "좋은 친구란?"
result = qa(query)


성경 잠언이 말하는 “좋은 친구”는 이런 사람입니다.

- 함께 있을 때 마음을 기쁘게 하고 생기를 주는 사람 (잠15:30)
- 생명을 주는 책망과 지혜로운 권고를 해 주고, 그런 권고를 서로 듣는 사람 (잠15:31; 잠27:9)
- 환난 날에 곁을 지키며 관계를 버리지 않는 가까운 이웃 같은 사람 (잠27:10)
- 무모함을 말리고 재앙을 피하도록 경고해 주는 신중한 사람 (잠27:12)
- 요란하고 배려 없는 방식이 아니라, 때와 방법을 헤아리는 사람 (잠27:14)
- 서로를 연마하여 더 선하고 지혜롭게 자라게 하는 사람 (잠27:17)
- 의로움을 따르고 하나님께 가까이 나아가 기도하는 사람 (잠15:29)

한마디로, 좋은 친구는 곁을 지키며 지혜롭게 권면하고, 서로를 더 나은 방향으로 다듬어 주는 충성된 이웃입니다.

In [30]:
query = "좋은 배우자란?"
result = qa(query)

좋은 배우자는 함께 의와 진실을 사랑하고, 가정을 살리며, 하나님을 의지하는 사람입니다. 주신 말씀들로 보면 이런 모습입니다.

- 의를 추구하고 악에서 멀리함, 기도하는 사람 (잠 15:29)
- 따뜻한 눈빛과 말로 마음을 기쁘게 하고, 좋은 소식과 격려를 전함 (잠 15:30)
- 훈계와 교정을 기꺼이 듣는 배우려는 태도(가르침에 귀 기울임) (잠 15:31)
- 거짓과 헛됨을 멀리하는 정직함 (잠 30:8)
- 물질에 치우치지 않는 절제와 만족을 구함(부하기도 가난하기도보다 필요한 양식) (잠 30:8-9)
- 재물을 의지하지 않고 하나님을 의지함(하나님의 말씀을 신뢰하고 더하지 않음) (잠 30:5-6; 11:28)
- 집을 해치지 않고 세우는 책임감과 지혜 (잠 11:29)
- 남을 비방하지 않고 말로 죄를 만들지 않음 (잠 30:10)

분별을 위해 스스로와 서로에게 물어보면 좋습니다.
- 우리는 서로의 조언과 교정을 겸손히 듣는가? (잠 15:31)
- 돈보다 의와 하나님을 먼저 두는가? (잠 11:28; 30:8-9)
- 우리의 말이 상대를 살리고 격려하는가? (잠 15:30)
- 거짓과 비방을 멀리하는가? (잠 30:8,10)

완벽한 사람은 없지만, 이런 방향으로 함께 자라가려는 사람이 좋은 배우자입니다. 그리고 나 역시 그런 사람이 되려는 결심이 관계를 복되게 합니다.